# simmodlr Results Export Validator
Cross-check UI-displayed numbers against the raw JSON results export.

**Set your file path in the next cell, then Run All.**

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from math import sqrt, isclose

# Set your results file path here
FILE = r"C:\Users\parki\Downloads\A&E-Batch 15-06-2026 19-17.json"

data = json.loads(Path(FILE).read_text(encoding="utf-8"))

## 1. Top-Level Structure

In [ ]:
print(f"Schema:     {data.get('schema', 'N/A')}")
print(f"Exported:   {data.get('exportedAt', 'N/A')}")
print(f"Status:     {data.get('status', 'N/A')}")
print(f"MetricsOnly:{data.get('metricsOnly', False)}")
print(f"\nTop-level keys: {list(data.keys())}")

## 2. Experiment Config

In [ ]:
exp = data.get("experiment", {})
print(f"Run Label:     {exp.get('runLabel', 'N/A')}")
print(f"Seed:          {exp.get('seed', 'N/A')}")
print(f"Replications:  {exp.get('replications', 'N/A')}")
print(f"Warm-up:       {exp.get('warmupPeriod', 0)}")
print(f"Max sim time:  {exp.get('maxSimTime', 'N/A')}")
print(f"Term. mode:    {exp.get('terminationMode', 'N/A')}")
if data.get('model'):
    print(f"Model:         {data['model'].get('name', 'N/A')}")

## 3. Per-Replication Summary
Compare each replication row displayed in the UI.

In [ ]:
reps = data.get("replications", [])
if not reps and data.get("results"):
    reps = [{"replicationIndex": 1, "seed": exp.get("seed"), "summary": data["results"].get("summary", {})}]

if reps:
    rows = []
    for r in reps:
        s = r.get("summary", {})
        rows.append({
            "Rep": r.get("replicationIndex"),
            "Seed": r.get("seed"),
            "Arrived": s.get("total", s.get("arrived", 0)),
            "Served": s.get("served", 0),
            "Reneged": s.get("reneged", 0),
            "Rate %": round(s.get("servedRatio", 0) * 100, 1),
            "AvgWait": round(s.get("avgWait", 0), 2),
            "AvgSvc": round(s.get("avgSvc", 0), 2),
            "Sojourn": round(s.get("avgSojourn", 0), 2),
            "TotalCost": round(s.get("totalCost", 0), 2),
            "Cost/Served": round(s.get("costPerServed", 0), 2),
            "WIP": round(s.get("avgWIP", 0), 2),
            "FinalTime": r.get("finalTime", ""),
        })
    df_reps = pd.DataFrame(rows)
    display(df_reps)
else:
    print("No replication data found.")

## 4. Aggregate Confidence Intervals
Cross-check: recompute CI from raw replication values vs stored CI.

In [ ]:
T_TABLE = {1:12.706,2:4.303,3:3.182,4:2.776,5:2.571,6:2.447,7:2.365,8:2.306,9:2.262,
           10:2.228,11:2.201,12:2.179,13:2.160,14:2.145,15:2.131,16:2.120,17:2.110,
           18:2.101,19:2.093,20:2.086,21:2.080,22:2.074,23:2.069,24:2.064,25:2.060,
           26:2.056,27:2.052,28:2.048,29:2.045,30:2.042}

def ci95(values):
    n = len(values)
    if n < 2:
        return {"n": n, "mean": values[0] if n else None, "lower": None, "upper": None}
    mean = sum(values) / n
    if n == 2:
        h = abs(values[0] - values[1]) * 0.5
        return {"n": n, "mean": mean, "lower": mean - h, "upper": mean + h}
    variance = sum((v - mean) ** 2 for v in values) / (n - 1)
    t = T_TABLE.get(n - 1, 1.96)
    half = t * sqrt(variance / n)
    return {"n": n, "mean": mean, "lower": mean - half, "upper": mean + half}

agg = data.get("aggregateStats", {})
if agg and reps:
    results_data = []
    for metric, stored_ci in agg.items():
        label = metric.replace("summary.", "")
        raw_vals = [r["summary"].get(label, r["summary"].get(metric, None)) for r in reps]
        raw_vals = [v for v in raw_vals if v is not None]
        if raw_vals:
            computed = ci95(raw_vals)
            match = isclose(computed["mean"], stored_ci.get("mean") or 0, rel_tol=0.01)
            results_data.append({
                "Metric": label,
                "Stored Mean": round(stored_ci.get("mean", 0) or 0, 4),
                "Computed Mean": round(computed["mean"], 4),
                "Stored Lower": round(stored_ci.get("lower", 0) or 0, 4),
                "Computed Lower": round(computed["lower"] or 0, 4),
                "Stored Upper": round(stored_ci.get("upper", 0) or 0, 4),
                "Computed Upper": round(computed["upper"] or 0, 4),
                "n": computed["n"],
                "Match": "OK" if match else "MISMATCH",
            })
    df_ci = pd.DataFrame(results_data)
    display(df_ci)
    mismatches = df_ci[df_ci["Match"] == "MISMATCH"]
    if len(mismatches):
        print(f"\nWARNING: {len(mismatches)} CI mismatch(es) found!")
    else:
        print("\nAll CI values match - stored vs computed.")
else:
    print("No aggregate stats or replications available for CI cross-check.")

## 5. Summary KPIs
Headline numbers shown in the UI Results Summary panel.

In [ ]:
summary = data.get("results", {}).get("summary", data.get("summary", {}))
if summary:
    kpis = {
        "Arrived": summary.get("total", summary.get("arrived")),
        "Served": summary.get("served"),
        "Recepted": summary.get("reneged"),
        "Recept Rate %": round((summary.get("renegeRate") or 0) * 100, 1) if summary.get("renegeRate") else None,
        "Completion Rate %": round((summary.get("servedRatio") or 0) * 100, 1),
        "Avg Wait": round(summary.get("avgWait") or 0, 2),
        "Avg Service": round(summary.get("avgSvc") or 0, 2),
        "Avg Sojourn": round(summary.get("avgSojourn") or 0, 2),
        "Avg Time in System": round(summary.get("avgTimeInSystem") or 0, 2),
        "Avg WIP": round(summary.get("avgWIP") or 0, 2),
        "Total Cost": round(summary.get("totalCost") or 0, 2),
        "Cost per Served": round(summary.get("costPerServed") or 0, 2),
        "Wait by Little": round(summary.get("avgWaitByLittle") or 0, 2),
        "Wait Discrepancy %": round(summary.get("waitDiscrepancy") or 0, 1),
    }
    df_kpi = pd.DataFrame(list(kpis.items()), columns=["KPI", "Value"])
    display(df_kpi)
else:
    print("No summary data.")

## 6. Quick Consistency Checks
Basic arithmetic validation.

In [ ]:
s = data.get("results", {}).get("summary", data.get("summary", {}))
arrived = s.get("total", s.get("arrived", 0)) or 0
served = s.get("served", 0) or 0
recepted = s.get("reneged", 0) or 0
ratio = s.get("servedRatio", 0) or 0
avgW = s.get("avgWait", 0) or 0
maxT = exp.get("maxSimTime", 1)

print(f"Arrived:        {arrived}")
print(f"Served:         {served}")
print(f"Recepted:       {recepted}")
print(f"Served+Reneged: {served + reneged:,.0f}  vs  Arrived: {arrived:,.0f}  -> {'MATCH' if isclose(served+reneged, arrived, abs_tol=1) else 'GAP (in-progress entities)'}")
print(f"Served/Arrived:  {served/arrived*100:.1f}%  vs  Stored Completion Rate: {ratio*100:.1f}%  -> {'MATCH' if isclose(served/arrived, ratio, abs_tol=0.01) else 'MISMATCH'}")

wip = s.get("avgWIP")
if wip and maxT:
    arrRate = arrived / maxT
    littleW = wip / arrRate if arrRate > 0 else 0
    disc = s.get("waitDiscrepancy", 0)
    print(f"\nLittle's Law:")
    print(f"  avgWIP / arrivalRate = {littleW:.2f}")
    print(f"  Stored avgWait       = {avgW:.2f}")
    print(f"  Stored discrepancy   = {disc}%")
    print(f"  -> {'MATCH' if abs(littleW - avgW) / max(avgW, 0.01) < 0.1 else 'WARNING'}")

## 7. Per-Queue Breakdown

In [ ]:
s = data.get("results", {}).get("summary", data.get("summary", {}))
perQueue = s.get("perQueue", {})
if perQueue:
    q_rows = []
    for qname, qdata in perQueue.items():
        q_rows.append({
            "Queue": qname,
            **{k: round(v, 2) if isinstance(v, float) else v for k, v in qdata.items()}
        })
    display(pd.DataFrame(q_rows).set_index("Queue"))

    q_names = [r["Queue"] for r in q_rows if r.get("maxDepth") or r.get("maxQueueLength")]
    q_depths = [r.get("maxDepth", r.get("maxQueueLength", 0)) for r in q_rows if r.get("maxDepth") or r.get("maxQueueLength")]
    if q_names:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.barh(q_names, q_depths, color="steelblue")
        ax.set_xlabel("Max Queue Depth")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("No per-queue breakdown data (perQueue is null).")
    
# Queue journeys (entity paths)
queueJourneys = s.get("queueJourneys", {})
if queueJourneys:
    print("\nQueue journey paths (top 10):")
    jr = sorted(queueJourneys.items(), key=lambda x: -x[1])[:10]
    for path, count in jr:
        print(f"  {path}: {count}")

## 8. Per-Resource (Server) Breakdown

In [ ]:
s = data.get("results", {}).get("summary", data.get("summary", {}))
perRes = s.get("perResource", {})
if perRes:
    r_rows = []
    for rname, rdata in perRes.items():
        r_rows.append({
            "Resource": rname,
            **{k: round(v, 2) if isinstance(v, float) else v for k, v in rdata.items()}
        })
    display(pd.DataFrame(r_rows).set_index("Resource"))
else:
    print("No per-resource data.")

## 9. Time Series (Queue Depth + Server Utilisation)
These feed the charts in the Results view.

In [ ]:
ts = data.get("results", {}).get("timeSeries", data.get("timeSeries", []))
if ts:
    print(f"{len(ts)} time-series snapshots")
    
    # Collect queue names
    queue_names = set()
    server_names = set()
    for snap in ts:
        queue_names.update(snap.get("byQueue", {}).keys())
        byT = snap.get("byType", {})
        server_names.update(k for k, v in byT.items() if "busy" in (v or {}) or "total" in (v or {}))
    
    # Queue depth line chart
    if queue_names:
        fig, ax = plt.subplots(figsize=(12, 4))
        times = [s.get("t", 0) for s in ts]
        for qname in sorted(queue_names):
            depths = [s.get("byQueue", {}).get(qname, {}).get("waiting", 0) or 0 for s in ts]
            ax.step(times, depths, where="post", label=qname, linewidth=1.5)
        ax.set_xlabel("Simulation Time")
        ax.set_ylabel("Entities Waiting")
        ax.legend(fontsize=8, loc="upper left")
        ax.set_title("Queue Depth Over Time")
        plt.tight_layout()
        plt.show()
    
    # Server utilisation line chart
    if server_names:
        fig, ax = plt.subplots(figsize=(12, 4))
        times = [s.get("t", 0) for s in ts]
        for sv in sorted(server_names):
            utils = []
            for s in ts:
                b = s.get("byType", {}).get(sv, {}).get("busy", 0) or 0
                t = s.get("byType", {}).get(sv, {}).get("total", 1) or 1
                utils.append((b / t) * 100 if t > 0 else 0)
            ax.step(times, utils, where="post", label=f"{sv} (% busy)", linewidth=1.5)
        ax.set_xlabel("Simulation Time")
        ax.set_ylabel("Utilisation %")
        ax.set_ylim(0, 110)
        ax.legend(fontsize=8, loc="upper left")
        ax.set_title("Server Utilisation Over Time")
        plt.tight_layout()
        plt.show()
else:
    print("No time-series data (run with Detailed output enabled).")

## 10. Wait Time Distributions
Histogram of wait times per queue - check against UI histograms.

In [ ]:
waitDist = data.get("results", {}).get("waitDist", data.get("waitDist", {}))
if waitDist:
    for qname, dist in waitDist.items():
        n = dist.get("n", 0)
        mean = dist.get("mean", 0)
        p50 = dist.get("p50")
        p95 = dist.get("p95")
        p99 = dist.get("p99")
        values = dist.get("values", [])
        print(f"\n{qname}: n={n}  mean={mean:.2f}  p50={p50}  p95={p95}  p99={p99}  samples={len(values)}")
        if values:
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.hist(values, bins=30, edgecolor="white", color="steelblue")
            ax.axvline(p50, color="green", linestyle="--", label=f"P50={p50}")
            ax.axvline(p95, color="orange", linestyle="--", label=f"P95={p95}")
            ax.axvline(p99, color="red", linestyle="--", label=f"P99={p99}")
            ax.set_xlabel("Wait Time")
            ax.set_ylabel("Count")
            ax.set_title(f"Wait Distribution - {qname}")
            ax.legend(fontsize=8)
            plt.tight_layout()
            plt.show()
else:
    print("No wait distribution data.")

## 11. Section Breakdown
If the model uses sections, this shows entity flow between them.

In [ ]:
s = data.get("results", {}).get("summary", data.get("summary", {}))
sections = s.get("sections", {})
if sections:
    sec_rows = []
    for sec_id, sec_data in sections.items():
        sec_rows.append({
            "Section": sec_id,
            "Count": sec_data.get("count", 0),
            "Avg Sojourn": round(sec_data.get("avgSojourn", 0), 2),
        })
    display(pd.DataFrame(sec_rows).set_index("Section"))

    # Journey paths
    journeys = s.get("journeys", {})
    if journeys:
        print("\nTop Journey Paths:")
        j_rows = []
        for path, count in sorted(journeys.items(), key=lambda x: -x[1]):
            j_rows.append({"Path": path, "Count": count})
        display(pd.DataFrame(j_rows))
else:
    print("No section data.")

## 12. Runtime Metrics

In [ ]:
rt = data.get("results", {}).get("runtimeMetrics", data.get("runtimeMetrics", {}))
if rt:
    df_rt = pd.DataFrame(list(rt.items()), columns=["Metric", "Value"])
    display(df_rt)
else:
    print("No runtime metrics.")

## 13. Explore Raw Data
Dump all top-level keys and inspect anything not yet covered.

In [ ]:
def explore(obj, prefix="", depth=0, max_depth=3):
    if depth > max_depth:
        return
    if isinstance(obj, dict):
        for k, v in obj.items():
            path = f"{prefix}.{k}" if prefix else k
            if isinstance(v, (dict, list)):
                kind = type(v).__name__
                size = len(v) if isinstance(v, (dict, list)) else ""
                print(f"  {path}: {kind}({size})")
            else:
                val = repr(v)[:80]
                print(f"  {path}: {val}")
    elif isinstance(obj, list) and len(obj) > 0:
        print(f"  {prefix}: list({len(obj)})")
        explore(obj[0], prefix + "[0]", depth + 1, max_depth)

print("Full data structure:")
explore(data)